# Stage 2.1 — Bigram (counting)

**No neural net. No gradients.** Just count which character follows which, normalize to probabilities, and sample.

This is the baseline every later stage measures itself against. The number to remember: **average NLL ≈ 2.45** on this dataset, with no smoothing. Every architecture from here on either beats it or it's a regression.

## What you'll ship

1. A 27×27 count matrix `N[i, j]` = how often character `j` followed character `i` in the training names
2. The probability table `P = N / N.sum(axis=1, keepdim=True)`
3. A sampler — draw a starting char from `P['.']`, sample the next, repeat until `.`
4. The negative-log-likelihood loss — the baseline number

## Reference

[Karpathy — The spelled-out intro to language modeling: bigrams](https://www.youtube.com/watch?v=PaCmpygFfXo) (~2 hrs). This notebook covers ~the first half of the video.

In [ ]:
import torch
import matplotlib.pyplot as plt
from pathlib import Path

print(f'torch {torch.__version__}  mps={torch.backends.mps.is_available()}')

## Load the data

In [ ]:
names_path = Path('data') / 'names.txt'
words = names_path.read_text().splitlines()
print(f'{len(words):,} names')
print(f'first 8: {words[:8]}')
print(f'shortest: {min(len(w) for w in words)} chars,  longest: {max(len(w) for w in words)} chars')
print(f'avg length: {sum(len(w) for w in words) / len(words):.2f}')

## Vocab

27 characters: 26 letters + `.` for both start-of-name and end-of-name. We use one token for both because the role is determined by *position*, not identity. (A name `emma` becomes `.emma.` — 5 bigrams: `.→e`, `e→m`, `m→m`, `m→a`, `a→.`.)

In [ ]:
chars = sorted(set(''.join(words)))   # 26 letters
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi['.'] = 0
itos = {i: c for c, i in stoi.items()}
vocab_size = len(stoi)
print(f'vocab size: {vocab_size}')
print(f'stoi: {stoi}')

## Build the bigram count matrix

`N[i, j]` = how often character `j` followed character `i`. We add `.` to both ends of each name so the model learns *what tends to start* a name and *what tends to end* one.

We add `1` to every cell (Laplace smoothing) so the log of a zero-probability transition doesn't blow up the NLL. Without this, any zero-frequency bigram in the test set gives `log(0) = -inf`.

In [ ]:
N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

print(f'total bigram observations: {N.sum().item():,}')
print(f'most-frequent starting char: {itos[N[0].argmax().item()]} (count {N[0].max().item():,})')
print(f'most-frequent ending char  : {itos[N[:, 0].argmax().item()]} (count {N[:, 0].max().item():,})')

## Visualize the count matrix

Karpathy renders this as a 27×27 heatmap. It's worth looking at — you can see the structure of English names. The `.` row at the top shows what letters start names; the `.` column on the left shows what letters end them.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 16))
ax.imshow(N, cmap='Blues')
for i in range(vocab_size):
    for j in range(vocab_size):
        chstr = itos[i] + itos[j]
        ax.text(j, i, chstr, ha='center', va='bottom', color='gray', fontsize=8)
        ax.text(j, i, N[i, j].item(), ha='center', va='top', color='gray', fontsize=7)
ax.set_xticks(range(vocab_size)); ax.set_xticklabels([itos[i] for i in range(vocab_size)])
ax.set_yticks(range(vocab_size)); ax.set_yticklabels([itos[i] for i in range(vocab_size)])
ax.set_title('Bigram counts: rows = current char, columns = next char')
ax.axis('off')
plt.tight_layout(); plt.show()

## Normalize counts → probabilities

`P[i, :]` is the probability distribution over next characters given current character `i`. Each row sums to 1.

We add `1` to every count before normalizing — Laplace smoothing. Without it, any unobserved bigram has probability 0, and `log(0)` is `-inf`, which corrupts the loss.

In [ ]:
P = (N + 1).float()
P /= P.sum(dim=1, keepdim=True)

# Sanity: every row sums to 1
row_sums = P.sum(dim=1)
print(f'row sums in [{row_sums.min():.6f}, {row_sums.max():.6f}]  (should be ~1.0)')
print(f'P[".", :] top 5 most likely starting chars:')
top5 = P[0].topk(5)
for prob, idx in zip(top5.values, top5.indices):
    print(f'  {itos[idx.item()]}  {prob.item():.4f}')

## Sample some names

Start at `.`, sample the next character from `P[current, :]`, advance, repeat until `.`. Use a fixed seed so the names are reproducible.

**What you should see**: short, name-shaped strings. Some are real-ish (`mor`, `kay`), some are garbage (`bryd`, `iiyazitlbpgu`). The model has captured *some* structure (vowels follow consonants, names are short) but not enough — the next stages fix this.

In [ ]:
def sample_name(P, itos, generator):
    out = []
    ix = 0  # start at '.'
    while True:
        ix = torch.multinomial(P[ix], num_samples=1, replacement=True, generator=generator).item()
        if ix == 0:
            return ''.join(out)
        out.append(itos[ix])

g = torch.Generator().manual_seed(2147483647)
print('10 samples from the bigram model:')
for _ in range(10):
    print(f'  {sample_name(P, itos, g)}')

## The baseline number — average NLL

**Negative log-likelihood** of the training set under our model, averaged per bigram. This is the number every later stage is graded against.

Formula: for each bigram `(ch1, ch2)` in every name, the log-probability under our model is `log P[stoi[ch1], stoi[ch2]]`. We sum these (negated) and divide by the count to get an average.

**Expected**: ≈ **2.45** (with Laplace smoothing of +1). The theoretical lower bound on a uniform model would be `log(27) ≈ 3.30`. So even just counting buys you ~0.85 nats of information per character.

In [ ]:
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        prob = P[stoi[ch1], stoi[ch2]]
        log_likelihood += torch.log(prob)
        n += 1

nll = -log_likelihood / n
print(f'total log-likelihood: {log_likelihood.item():,.4f}')
print(f'total bigrams: {n:,}')
print(f'average NLL: {nll.item():.4f}')
print(f'\nThis is the bigram baseline. Stage 2.2 should ~match it (same model, neural net form).')
print(f'Stage 2.3 (Bengio MLP) should beat it (~2.10).')

## End-of-stage check

- [ ] Average NLL is ≈ 2.45
- [ ] You can read 10 samples without thinking "this is total noise"
- [ ] You can explain why `log(0)` would corrupt the NLL without smoothing

Append one line to `NOTES.md` describing what surprised you about the bigram baseline. (Common one: "how much structure a simple count table captures vs. how little it does for names that aren't 4-letter common English.")

Next: Stage 2.2 (`02_bigram_nn.ipynb`) — same task, neural-net form. The NLL should land at roughly the same number, but now via gradient descent on a learned weight matrix. That's the bridge from "counting" to "trained model".